In [ ]:
import os
import neuprint as neu
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

c = neu.Client('https://neuprint.janelia.org',
               dataset='male-cns:v1.0',
               token=os.environ['NEUPRINT_TOKEN'])
DATA = Path.home() / 'arm_simulation' / 'brain' / 'data'
DATA.mkdir(parents=True, exist_ok=True)
print("Connected to:", c.dataset)

neurons = pd.read_parquet(DATA / 'neurons_male-cns_v1.0.parquet')
CLASS, SUBCLASS, NT, SIDE = 'superclass', 'class', 'predictedNt', 'somaSide'
props = c.fetch_custom("""
    MATCH (n:Neuron) WITH n LIMIT 2000
    UNWIND keys(n) AS k RETURN DISTINCT k AS property
""")
available = set(props['property'])

print("Ready:", c.dataset, "|", len(neurons), "neurons loaded")

In [ ]:
for g in ['descending_neuron', 'vnc_motor']:
    sub = neurons[neurons['cls'] == g]
    print(f"\n=== {g}: {sub['type'].nunique()} distinct types ===")
    display(sub['type'].value_counts().head(25).to_frame('neurons'))

keywords = ['neuromere', 'nerve', 'target', 'segment', 'hemilineage',
            'group', 'region', 'leg', 'muscle', 'synonym', 'entry', 'exit']
hits = sorted(p for p in available if any(k in p.lower() for k in keywords))
print("\nCandidate anatomy fields:", hits)

In [ ]:
# Extra anatomy fields for sensory and motor neurons
extra = c.fetch_custom("""
    MATCH (n:Neuron)
    WHERE n.superclass IN ['vnc_motor', 'cb_sensory']
    RETURN n.bodyId AS bodyId, n.somaNeuromere AS neuromere,
           n.exitNerve AS exitNerve, n.entryNerve AS entryNerve
""")
df = neurons.merge(extra, on='bodyId', how='left')

# Check the values before filtering on them
mn_all = df[df.cls == 'vnc_motor']
print("Motor neuron neuromeres:\n", mn_all['neuromere'].value_counts(dropna=False), "\n")
print("Motor neuron exit nerves:\n", mn_all['exitNerve'].value_counts(dropna=False).head(15), "\n")

# OUTPUT group: front-leg (T1) motor neurons, excluding abdominal/flight types
non_leg = mn_all['type'].astype(str).str.match(r'^(MNad|DLM|MNwm|MNhl|MNnm)')
MN = mn_all[mn_all['neuromere'].astype(str).str.contains('T1') & ~non_leg]
print(f"Front-leg motor neurons: {len(MN)}")
display(MN['type'].value_counts().head(20).to_frame('neurons'))

# INPUT group: head mechanosensors (mostly antennal)
S = df[(df.cls == 'cb_sensory') & (df.subcls == 'mechanosensory')]
print(f"\nHead mechanosensory neurons: {len(S)}")
display(S['type'].value_counts().head(15).to_frame('neurons'))

DN = df[df.cls == 'descending_neuron']
print(f"\nDescending neurons (all): {len(DN)}")

In [ ]:
# INPUT: Johnston's organ C and E neurons (antennal grooming)
S = df[(df.cls == 'cb_sensory') & df['type'].astype(str).str.match(r'^JO-(C|E)')]
print(f"JO-C/E neurons: {len(S)}")
display(S['type'].value_counts().to_frame('neurons'))

# OUTPUT: T1 motor neurons that leave through the front-leg nerve
MN = mn_all[mn_all['neuromere'].astype(str).str.contains('T1') & (mn_all['exitNerve'] == 'ProLN')]
print(f"\nFront-leg (ProLN) motor neurons: {len(MN)}")
display(MN['side'].value_counts().to_frame('neurons'))
display(MN['type'].value_counts().head(20).to_frame('neurons'))

In [ ]:
from neuprint import fetch_adjacencies, NeuronCriteria as NC

post_total = neurons.set_index('bodyId')['post']

def edges(src=None, tgt=None):
    """Connections with ≥5 synapses, as a fraction of the target's total input."""
    _, conn = fetch_adjacencies(
        NC(bodyId=list(src)) if src is not None else None,
        NC(bodyId=list(tgt)) if tgt is not None else None,
        min_total_weight=5)
    e = conn.groupby(['bodyId_pre', 'bodyId_post'], as_index=False)['weight'].sum()
    e['frac'] = e['weight'] / e['bodyId_post'].map(post_total)
    return e

S_ids, DN_ids, MN_ids = set(S.bodyId), set(DN.bodyId), set(MN.bodyId)

# Input side: sensors -> DN directly, or sensors -> X -> DN
e1 = edges(src=S_ids)                                   # everything sensors talk to
e2 = edges(src=set(e1.bodyId_post), tgt=DN_ids)         # those partners -> DNs
two_in = e1.merge(e2, left_on='bodyId_post', right_on='bodyId_pre', suffixes=('_1', '_2'))
two_in['path'] = two_in.frac_1 * two_in.frac_2
in_score = (two_in.groupby('bodyId_post_2')['path'].sum()
            .add(e1[e1.bodyId_post.isin(DN_ids)].groupby('bodyId_post')['frac'].sum(), fill_value=0))

# Output side: DN -> MN directly, or DN -> Y -> MN
e3 = edges(tgt=MN_ids)                                  # everything that drives the MNs
e4 = edges(src=DN_ids, tgt=set(e3.bodyId_pre))          # DNs -> those premotor neurons
two_out = e4.merge(e3, left_on='bodyId_post', right_on='bodyId_pre', suffixes=('_1', '_2'))
two_out['path'] = two_out.frac_1 * two_out.frac_2
out_score = (two_out.groupby('bodyId_pre_1')['path'].sum()
             .add(e3[e3.bodyId_pre.isin(DN_ids)].groupby('bodyId_pre')['frac'].sum(), fill_value=0))

rank = pd.DataFrame({'in_score': in_score, 'out_score': out_score}).fillna(0)
rank['score'] = np.sqrt(rank.in_score * rank.out_score)   # must be strong on BOTH sides
rank = rank.join(DN.set_index('bodyId')[['type', 'instance']]).sort_values('score', ascending=False)

display(rank.head(25))
print("DNs with any path on both sides:", (rank.score > 0).sum())

In [ ]:
top_dn_ids = rank[rank.score > 0].head(20).index
DN_top = DN[DN.bodyId.isin(top_dn_ids)].merge(
    rank[['in_score', 'out_score', 'score']], left_on='bodyId', right_index=True)

cols = ['bodyId', 'type', 'instance', 'side', 'nt']
shortlist = pd.concat([
    S[cols].assign(group='input_JO-CE'),
    DN_top[cols + ['in_score', 'out_score', 'score']].assign(group='middle_descending'),
    MN[cols].assign(group='output_T1_leg_MN'),
], ignore_index=True)

shortlist.to_csv(DATA / 'shortlist_week1.csv', index=False)
rank.to_csv(DATA / 'dn_ranking_week1.csv')
print(shortlist['group'].value_counts())

In [ ]:
# Candidates: top 10 by input, by output, and by combined score
pick = (set(rank.nlargest(10, 'in_score').index)
        | set(rank.nlargest(10, 'out_score').index)
        | set(rank.nlargest(10, 'score').index))
types = rank.loc[list(pick), 'type'].unique()          # include both L and R of each type

DN_mid = DN[DN['type'].isin(types)].merge(
    rank[['in_score', 'out_score', 'score']], left_on='bodyId', right_index=True)
DN_mid['role'] = np.where(DN_mid.in_score > 3 * DN_mid.out_score, 'input-side',
                 np.where(DN_mid.out_score > 3 * DN_mid.in_score, 'output-side', 'bridge'))

cols = ['bodyId', 'type', 'instance', 'side', 'nt']
shortlist = pd.concat([
    S[cols].assign(group='input_JO-CE'),
    DN_mid[cols + ['in_score', 'out_score', 'score', 'role']].assign(group='middle_descending'),
    MN[cols].assign(group='output_T1_leg_MN'),
], ignore_index=True)

shortlist.to_csv(DATA / 'shortlist_week1.csv', index=False)
print(shortlist['group'].value_counts())
display(DN_mid.sort_values('score', ascending=False)[
    ['type', 'instance', 'nt', 'in_score', 'out_score', 'score', 'role']])

In [ ]:
props = c.fetch_custom("""
    MATCH (n:Neuron) WITH n LIMIT 2000
    UNWIND keys(n) AS k
    RETURN k AS property, count(*) AS n ORDER BY n DESC
""")
available = set(props['property'])
print(sorted(available))

In [ ]:
def pick(candidates):
    return next((p for p in candidates if p in available), None)

CLASS    = pick(['superclass', 'class'])
SUBCLASS = pick(['class', 'subclass']) if CLASS == 'superclass' else pick(['subclass'])
NT       = pick(['predictedNt', 'consensusNt', 'celltypePredictedNt', 'predicted_nt'])
SIDE     = pick(['somaSide', 'rootSide', 'side'])

print(f"class={CLASS}  subclass={SUBCLASS}  nt={NT}  side={SIDE}")

In [ ]:
from pathlib import Path
DATA = Path.home() / 'arm_simulation' / 'brain' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

fields = {'bodyId': 'bodyId', 'type': 'type', 'instance': 'instance',
          'pre': 'pre', 'post': 'post',
          'cls': CLASS, 'subcls': SUBCLASS, 'nt': NT, 'side': SIDE}
ret = ", ".join(f"n.{p} AS {alias}" for alias, p in fields.items() if p)

neurons = c.fetch_custom(f"MATCH (n:Neuron) RETURN {ret}")
neurons.to_parquet(f'{DATA}/neurons_male-cns_v1.0.parquet')
print(len(neurons), "neurons")
neurons.head()

In [ ]:
counts = neurons['cls'].value_counts(dropna=False)
display(counts.to_frame('neurons'))

counts.plot.barh(figsize=(8, 6), logx=True)
plt.xlabel('number of neurons (log scale)')
plt.title('Neurons by class, male-CNS v1.0')
plt.tight_layout(); plt.show()

groups = ['vnc_sensory', 'cb_sensory', 'descending_neuron', 'vnc_motor', 'cb_motor']

for g in groups:
    sub = neurons[neurons['cls'] == g]
    print(f"\n=== {g}: {len(sub)} neurons ===")
    display(sub['subcls'].value_counts(dropna=False).head(15).to_frame('neurons'))

In [ ]:
nt_counts = neurons['nt'].value_counts(dropna=False)
display(nt_counts.to_frame('neurons'))

# Neurotransmitter mix within each class, as fractions
mix = pd.crosstab(neurons['cls'], neurons['nt'], normalize='index').round(2)
display(mix)

nt_counts.plot.bar(figsize=(7, 4))
plt.ylabel('neurons'); plt.title('Predicted neurotransmitter')
plt.tight_layout(); plt.show()

In [ ]:
# Per-neuron totals of input (post) and output (pre) synapses
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col in zip(ax, ['post', 'pre']):
    a.hist(neurons[col].clip(lower=1), bins=np.logspace(0, 6, 60))
    a.set_xscale('log'); a.set_yscale('log')
    a.set_xlabel(f'{col} synapses per neuron'); a.set_ylabel('neurons')
plt.tight_layout(); plt.show()

# Per-connection weights, counted on the server so the whole edge table isn't downloaded
w = c.fetch_custom("""
    MATCH (:Neuron)-[e:ConnectsTo]->(:Neuron)
    RETURN e.weight AS weight, count(*) AS n_connections
    ORDER BY weight
""")
w.to_parquet(f'{DATA}/weight_histogram.parquet')

plt.figure(figsize=(6, 4))
plt.loglog(w['weight'], w['n_connections'], '.')
plt.xlabel('synapses per connection'); plt.ylabel('number of connections')
plt.title('Connection weight distribution'); plt.show()

print("Total connections:", w['n_connections'].sum())
print("Connections with ≥5 synapses:", w.loc[w.weight >= 5, 'n_connections'].sum())